In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

In [ ]:
df_game_info = pd.read_csv("Data/exposure_and_game_info/nba_games_2025-26.csv")
display(df_game_info.head(3))

df_palyer_info = pd.read_csv("Data/exposure_and_game_info/nba_players_stats_2025-26.csv")
display(df_palyer_info.head(3))

In [ ]:
# enrichissement de la base de données des matchs, on vient rajouter le nombre de victoire et défaite avant match de chaque équipe et leur série de victoire/défaite en cours

# Trier par date
df_game_info = df_game_info.sort_values(by='GAME_DATE', ascending=True).reset_index(drop=True)

# Déterminer le gagnant de chaque match
df_game_info['HOME_WIN'] = (df_game_info['HOME_PTS'] > df_game_info['AWAY_PTS']).astype(int)
df_game_info['AWAY_WIN'] = (df_game_info['AWAY_PTS'] > df_game_info['HOME_PTS']).astype(int)

# Initialiser les colonnes
df_game_info['HOME_TEAM_wins_before'] = 0
df_game_info['HOME_TEAM_losses_before'] = 0
df_game_info['AWAY_TEAM_wins_before'] = 0
df_game_info['AWAY_TEAM_losses_before'] = 0
df_game_info['HOME_TEAM_streak_before'] = 0  # Positif = victoires, Négatif = défaites
df_game_info['AWAY_TEAM_streak_before'] = 0

# Dictionnaire pour mapper les équipes à leur conférence
# Conférence EST
east_teams = {
    1610612738,  # Boston Celtics
    1610612751,  # Brooklyn Nets
    1610612752,  # New York Knicks
    1610612755,  # Philadelphia 76ers
    1610612761,  # Toronto Raptors
    1610612741,  # Chicago Bulls
    1610612739,  # Cleveland Cavaliers
    1610612765,  # Detroit Pistons
    1610612754,  # Indiana Pacers
    1610612749,  # Milwaukee Bucks
    1610612737,  # Atlanta Hawks
    1610612766,  # Charlotte Hornets
    1610612748,  # Miami Heat
    1610612753,  # Orlando Magic
    1610612764,  # Washington Wizards
}

# Créer les variables de conférence binaires (1 = EST, 0 = OUEST)
df_game_info['HOME_TEAM_is_east'] = df_game_info['HOME_TEAM_ID'].isin(east_teams).astype(int)
df_game_info['AWAY_TEAM_is_east'] = df_game_info['AWAY_TEAM_ID'].isin(east_teams).astype(int)

# Dictionnaire pour suivre les stats de chaque équipe
team_stats = {}

# Parcourir chaque match dans l'ordre chronologique
for idx, row in df_game_info.iterrows():
    home_team = row['HOME_TEAM_ID']
    away_team = row['AWAY_TEAM_ID']
    
    # Initialiser les équipes si première apparition
    if home_team not in team_stats:
        team_stats[home_team] = {'wins': 0, 'losses': 0, 'streak': 0}
    if away_team not in team_stats:
        team_stats[away_team] = {'wins': 0, 'losses': 0, 'streak': 0}
    
    # Enregistrer les stats AVANT le match
    df_game_info.at[idx, 'HOME_TEAM_wins_before'] = team_stats[home_team]['wins']
    df_game_info.at[idx, 'HOME_TEAM_losses_before'] = team_stats[home_team]['losses']
    df_game_info.at[idx, 'AWAY_TEAM_wins_before'] = team_stats[away_team]['wins']
    df_game_info.at[idx, 'AWAY_TEAM_losses_before'] = team_stats[away_team]['losses']
    
    # Séries
    df_game_info.at[idx, 'HOME_TEAM_streak_before'] = team_stats[home_team]['streak']
    df_game_info.at[idx, 'AWAY_TEAM_streak_before'] = team_stats[away_team]['streak']
    
    # Mettre à jour les stats APRÈS le match
    if row['HOME_WIN'] == 1:
        # Home team gagne
        team_stats[home_team]['wins'] += 1
        team_stats[away_team]['losses'] += 1
        
        # Mettre à jour les séries
        if team_stats[home_team]['streak'] >= 0:
            team_stats[home_team]['streak'] += 1
        else:
            team_stats[home_team]['streak'] = 1
        
        if team_stats[away_team]['streak'] <= 0:
            team_stats[away_team]['streak'] -= 1
        else:
            team_stats[away_team]['streak'] = -1
    else:
        # Away team gagne
        team_stats[home_team]['losses'] += 1
        team_stats[away_team]['wins'] += 1
        
        # Mettre à jour les séries
        if team_stats[home_team]['streak'] <= 0:
            team_stats[home_team]['streak'] -= 1
        else:
            team_stats[home_team]['streak'] = -1
        
        if team_stats[away_team]['streak'] >= 0:
            team_stats[away_team]['streak'] += 1
        else:
            team_stats[away_team]['streak'] = 1

# Ajouter le nombre total de matchs joués avant
df_game_info['HOME_TEAM_games_before'] = (
    df_game_info['HOME_TEAM_wins_before'] + df_game_info['HOME_TEAM_losses_before']
)
df_game_info['AWAY_TEAM_games_before'] = (
    df_game_info['AWAY_TEAM_wins_before'] + df_game_info['AWAY_TEAM_losses_before']
)

# Calculer le win percentage avant le match
df_game_info['HOME_TEAM_winpct_before'] = df_game_info.apply(
    lambda x: x['HOME_TEAM_wins_before'] / x['HOME_TEAM_games_before'] 
    if x['HOME_TEAM_games_before'] > 0 else 0.5, axis=1
)
df_game_info['AWAY_TEAM_winpct_before'] = df_game_info.apply(
    lambda x: x['AWAY_TEAM_wins_before'] / x['AWAY_TEAM_games_before'] 
    if x['AWAY_TEAM_games_before'] > 0 else 0.5, axis=1
)

# Créer des colonnes lisibles pour les séries
df_game_info['HOME_TEAM_streak_label'] = df_game_info['HOME_TEAM_streak_before'].apply(
    lambda x: f"W{x}" if x > 0 else (f"L{abs(x)}" if x < 0 else "-")
)
df_game_info['AWAY_TEAM_streak_label'] = df_game_info['AWAY_TEAM_streak_before'].apply(
    lambda x: f"W{x}" if x > 0 else (f"L{abs(x)}" if x < 0 else "-")
)

In [ ]:
df_games_unique = df_game_info.drop_duplicates(['GAME_ID']).copy()

cols_home = [c for c in df_games_unique.columns if "HOME_" in c]
df_home = df_games_unique[['GAME_ID', 'GAME_DATE'] + cols_home].copy()
rename_dict_home = {c: c.replace('HOME_', '') for c in cols_home}
rename_dict_home['HOME_TEAM_ID'] = 'TEAM_ID'
df_home = df_home.rename(columns=rename_dict_home)

cols_away = [c for c in df_games_unique.columns if "AWAY_" in c]
df_away = df_games_unique[['GAME_ID', 'GAME_DATE'] + cols_away].copy()
rename_dict_away = {c: c.replace('AWAY_', '') for c in cols_away}
rename_dict_away['AWAY_TEAM_ID'] = 'TEAM_ID'
df_away = df_away.rename(columns=rename_dict_away)

df_team_stats = pd.concat([df_home, df_away], ignore_index=True).sort_values(by='GAME_DATE')
df_team_stats = df_team_stats.sort_values(by=['TEAM_ID', 'GAME_DATE'])

# Exclure les colonnes de victoires/défaites/classement/séries/IDs du rolling
exclude_cols = [
    'TEAM_wins_before', 'TEAM_losses_before', 
    'TEAM_games_before', 'TEAM_winpct_before',
    'TEAM_conf_rank_before', 'TEAM_league_rank_before',
    'TEAM_streak_before', 'WIN',
    'TEAM_ID', 'GAME_ID'  # Exclusion des IDs
]

# On identifie les colonnes numériques sur lesquelles faire la moyenne
# en excluant les colonnes de contexte de match et les IDs
cols_to_roll = [c for c in df_team_stats.select_dtypes(include='number').columns 
                if c not in exclude_cols]

df_rolling = df_team_stats.groupby('TEAM_ID')[cols_to_roll].apply(
    lambda x: x.shift(1).rolling(window=5, min_periods=1).mean()
).reset_index(level=0, drop=True)

df_rolling = df_rolling.add_suffix('_last5')
df_stats_calculated = pd.concat([df_team_stats[['GAME_ID', 'TEAM_ID']], df_rolling], axis=1)

df_final = df_game_info.copy()
df_final = df_final.merge(
    df_stats_calculated,
    left_on=['HOME_TEAM_ID', 'GAME_ID'],
    right_on=['TEAM_ID', 'GAME_ID'],
    how='left'
).drop(columns=['TEAM_ID'])

# Renommer les colonnes ajoutées pour préciser qu'il s'agit de HOME
new_cols = df_rolling.columns
rename_dict_final_home = {c: f"HOME_{c}" for c in new_cols}
df_final = df_final.rename(columns=rename_dict_final_home)

# 2. Merge pour l'équipe à l'Extérieur (AWAY)
df_final = df_final.merge(
    df_stats_calculated,
    left_on=['AWAY_TEAM_ID', 'GAME_ID'],
    right_on=['TEAM_ID', 'GAME_ID'],
    how='left'
).drop(columns=['TEAM_ID'])

# Renommer les colonnes ajoutées pour préciser qu'il s'agit de AWAY
rename_dict_final_away = {c: f"AWAY_{c}" for c in new_cols}
df_final = df_final.rename(columns=rename_dict_final_away)

In [ ]:
df_final['NB_GAMES_THIS_DAY'] = df_final.groupby('GAME_DATE')['GAME_ID'].transform('count')